*Practice Time based quries*

In [135]:
import duckdb

con = duckdb.connect("../sql/taxi_project")

In [136]:
con.sql("""
    SHOW TABLES;
""")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ taxi_clean      │
│ yellow_taxi_raw │
│ zone_lookup     │
└─────────────────┘

In [137]:
con.sql("""
    SELECT COUNT(*) 
    FROM taxi_clean;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      2816365 │
└──────────────┘

In [138]:
con.sql("""
    SELECT * FROM zone_lookup limit 1;
    SELECT * FROM taxi_clean limit 1;
""")

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-01 00:18:38 │ 2025-01-01 00:26:59 │               1 │           1.6 │            229 │            237 │        10.0 │        3.0 │
└─────────────────────┴─────────────────────┴─────────────────┴───────────────┴────────────────┴────────────────┴─────────────┴────────────┘

In [139]:
con.sql("""
    SELECT * FROM zone_lookup limit 1;""")

┌────────────┬─────────┬────────────────┬──────────────┐
│ LocationID │ Borough │      Zone      │ service_zone │
│   int64    │ varchar │    varchar     │   varchar    │
├────────────┼─────────┼────────────────┼──────────────┤
│          1 │ EWR     │ Newark Airport │ EWR          │
└────────────┴─────────┴────────────────┴──────────────┘

*Trips by day*

In [140]:
con.sql("""
    SELECT DAYNAME(DATE(pickup_datetime)) AS Day, COUNT(*) As total_trips
        FROM taxi_clean 
        GROUP BY Day
        ORDER BY total_trips
        DESC;
""")

┌───────────┬─────────────┐
│    Day    │ total_trips │
│  varchar  │    int64    │
├───────────┼─────────────┤
│ Thursday  │      499762 │
│ Friday    │      474504 │
│ Wednesday │      467267 │
│ Saturday  │      386606 │
│ Tuesday   │      374708 │
│ Sunday    │      310255 │
│ Monday    │      303263 │
└───────────┴─────────────┘

In [141]:
con.sql("""
    SELECT DATE(pickup_datetime) AS Day, COUNT(*) As total_trips
        FROM taxi_clean 
        GROUP BY Day
        ORDER BY total_trips
        DESC;
""")

┌────────────┬─────────────┐
│    Day     │ total_trips │
│    date    │    int64    │
├────────────┼─────────────┤
│ 2025-01-16 │      109757 │
│ 2025-01-23 │      106262 │
│ 2025-01-30 │      106202 │
│ 2025-01-25 │      105571 │
│ 2025-01-15 │      102728 │
│ 2025-01-24 │      101708 │
│ 2025-01-14 │      100011 │
│ 2025-01-09 │       99988 │
│ 2025-01-29 │       99420 │
│ 2025-01-11 │       99358 │
│     ·      │         ·   │
│     ·      │         ·   │
│     ·      │         ·   │
│ 2025-01-27 │       79684 │
│ 2025-01-26 │       79423 │
│ 2025-01-02 │       77553 │
│ 2025-01-19 │       77528 │
│ 2025-01-06 │       72879 │
│ 2025-01-05 │       72223 │
│ 2025-01-01 │       70384 │
│ 2025-01-20 │       65764 │
│ 2024-12-31 │          21 │
│ 2025-02-01 │           1 │
└────────────┴─────────────┘
  33 rows        2 columns
  (20 shown)               

*Trips by hour of day*

In [142]:
con.sql("""
    SELECT EXTRACT(HOUR FROM pickup_datetime) AS Hour, COUNT(*) As total_trips_by_hour
        FROM taxi_clean
        GROUP BY Hour
        ORDER BY total_trips_by_hour
        DESC
        LIMIT 10;
""")

┌───────┬─────────────────────┐
│ Hour  │ total_trips_by_hour │
│ int64 │        int64        │
├───────┼─────────────────────┤
│    17 │              206914 │
│    18 │              205923 │
│    16 │              190700 │
│    15 │              188160 │
│    14 │              178779 │
│    19 │              174590 │
│    13 │              165924 │
│    21 │              161481 │
│    20 │              157882 │
│    12 │              157073 │
└───────┴─────────────────────┘
  10 rows           2 columns

*Revenue by day*

In [143]:
con.sql("""
    SELECT DAYNAME(pickup_datetime) AS Day, ROUND(SUM(fare_amount)) as total_revenue
        FROM taxi_clean
        GROUP BY Day
        ORDER BY total_revenue
        DESC;
""")

┌───────────┬───────────────┐
│    Day    │ total_revenue │
│  varchar  │    double     │
├───────────┼───────────────┤
│ Thursday  │     9043424.0 │
│ Friday    │     8532111.0 │
│ Wednesday │     8325662.0 │
│ Saturday  │     6600927.0 │
│ Monday    │     6600422.0 │
│ Tuesday   │     6565249.0 │
│ Sunday    │     5738760.0 │
└───────────┴───────────────┘

*Average fare by weekday*

In [144]:
con.sql("""
    SELECT DAYNAME(pickup_datetime) as Day, ROUND(AVG(fare_amount)) as average_fare
        FROM taxi_clean
        GROUP BY Day
        ORDER BY average_fare
        DESC;
""")

┌───────────┬──────────────┐
│    Day    │ average_fare │
│  varchar  │    double    │
├───────────┼──────────────┤
│ Monday    │         22.0 │
│ Sunday    │         18.0 │
│ Tuesday   │         18.0 │
│ Thursday  │         18.0 │
│ Wednesday │         18.0 │
│ Friday    │         18.0 │
│ Saturday  │         17.0 │
└───────────┴──────────────┘

*Average tip by hour*

In [145]:
con.sql("""
    SELECT EXTRACT(HOUR FROM pickup_datetime) as Hour, ROUND(AVG(tip_amount)) as avg_tip
        FROM taxi_clean
        GROUP BY Hour 
        ORDER BY avg_tip
        DESC
        LIMIT 10;
""")

┌───────┬─────────┐
│ Hour  │ avg_tip │
│ int64 │ double  │
├───────┼─────────┤
│     0 │     4.0 │
│    14 │     4.0 │
│     5 │     4.0 │
│    16 │     4.0 │
│    15 │     4.0 │
│    18 │     4.0 │
│    17 │     4.0 │
│    19 │     4.0 │
│     4 │     4.0 │
│    20 │     4.0 │
└───────┴─────────┘
      10 rows    

*Busiest pickup hour per borough*

In [146]:
con.sql("""
    WITH rides_by_hour AS (
    SELECT EXTRACT(HOUR FROM t.pickup_datetime) as Hour, 
        z.Borough , 
        COUNT(*) as total_rides
    FROM taxi_clean as t
    INNER JOIN zone_lookup as z
        ON t.pu_Location_id = z.LocationID
    GROUP BY Hour, z.Borough
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY Borough
                ORDER BY total_rides DESC
            ) AS rank
        FROM rides_by_hour
    )
        SELECT
            *
        FROM ranked;
""")

┌───────┬───────────────┬─────────────┬───────┐
│ Hour  │    Borough    │ total_rides │ rank  │
│ int64 │    varchar    │    int64    │ int64 │
├───────┼───────────────┼─────────────┼───────┤
│    13 │ Staten Island │          18 │     1 │
│    15 │ Staten Island │          11 │     2 │
│    14 │ Staten Island │          10 │     3 │
│    16 │ Staten Island │           4 │     4 │
│    11 │ Staten Island │           3 │     5 │
│    10 │ Staten Island │           3 │     6 │
│    12 │ Staten Island │           3 │     7 │
│    17 │ Staten Island │           2 │     8 │
│     8 │ Staten Island │           2 │     9 │
│     5 │ Staten Island │           2 │    10 │
│     · │   ·           │           · │     · │
│     · │   ·           │           · │     · │
│     · │   ·           │           · │     · │
│    18 │ Bronx         │         125 │    15 │
│    19 │ Bronx         │          99 │    16 │
│    20 │ Bronx         │          75 │    17 │
│    21 │ Bronx         │          72 │ 

In [147]:
con.sql("""
    WITH rides_by_hour AS (
    SELECT EXTRACT(HOUR FROM t.pickup_datetime) as Hour, 
        z.Borough , 
        COUNT(*) as total_rides
    FROM taxi_clean as t
    INNER JOIN zone_lookup as z
        ON t.pu_Location_id = z.LocationID
    GROUP BY Hour, z.Borough
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY Borough
                ORDER BY total_rides DESC
            ) AS rank
        FROM rides_by_hour
    )
        SELECT
            Borough,
            Hour,
            total_rides
        FROM ranked
        WHERE rank = 1
        ORDER BY total_rides DESC;
""")

┌───────────────┬───────┬─────────────┐
│    Borough    │ Hour  │ total_rides │
│    varchar    │ int64 │    int64    │
├───────────────┼───────┼─────────────┤
│ Manhattan     │    18 │      190283 │
│ Queens        │    16 │       18570 │
│ Brooklyn      │     9 │        2201 │
│ Bronx         │     9 │         580 │
│ Staten Island │    13 │          18 │
│ EWR           │    16 │          15 │
└───────────────┴───────┴─────────────┘

*Weekend vs weekday revenue*

In [148]:
con.sql("""
    SELECT 
        CASE
            WHEN DAYNAME(pickup_datetime) IN ('Saturday', 'Sunday')
                THEN 'Weekend'
            ELSE 'Weekday'
        END AS 'day_type',
        ROUND(SUM(fare_amount), 2) AS revenue,
    FROM taxi_clean
    GROUP BY day_type
    ORDER BY revenue DESC;
""")

┌──────────┬─────────────┐
│ day_type │   revenue   │
│ varchar  │   double    │
├──────────┼─────────────┤
│ Weekday  │ 39066867.14 │
│ Weekend  │ 12339687.08 │
└──────────┴─────────────┘

*Weekend vs weekday revenue with percentage*

In [149]:
con.sql("""
    WITH revenue_by_day_type AS (
        SELECT 
            CASE
                WHEN DAYNAME(pickup_datetime) IN ('Saturday', 'Sunday')
                    THEN 'Weekend'
                ELSE 'Weekday'
            END AS 'day_type',
            SUM(fare_amount) AS revenue
        FROM taxi_clean
        GROUP BY day_type
    )
    SELECT 
        day_type,
        ROUND(revenue, 2) AS revenue,
        ROUND(revenue / SUM(revenue) OVER() * 100, 2) AS revenu_perc
    FROM revenue_by_day_type
    ORDER BY revenue DESC;
""")

┌──────────┬─────────────┬─────────────┐
│ day_type │   revenue   │ revenu_perc │
│ varchar  │   double    │   double    │
├──────────┼─────────────┼─────────────┤
│ Weekday  │ 39066867.14 │        76.0 │
│ Weekend  │ 12339687.08 │        24.0 │
└──────────┴─────────────┴─────────────┘

*Morning rush vs evening rush*

In [150]:
con.sql("""
    SELECT
        CASE
            WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 6 AND 11
                THEN 'morning_Rush'
            WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 16 AND 19
                THEN 'evening_rush'
        END AS rush_type,
        COUNT(*) as number_of_rides
    FROM taxi_clean
    WHERE EXTRACT(HOUR FROM pickup_datetime) BETWEEN 6 AND 11
    OR EXTRACT(HOUR FROM pickup_datetime) BETWEEN 16 AND 19
    GROUP BY rush_type
    ORDER BY number_of_rides DESC;
""")

┌──────────────┬─────────────────┐
│  rush_type   │ number_of_rides │
│   varchar    │      int64      │
├──────────────┼─────────────────┤
│ evening_rush │          778127 │
│ morning_Rush │          615285 │
└──────────────┴─────────────────┘

*Trips grouped into 15-minute windows*

In [151]:
con.sql("""
    SELECT
        time_bucket(INTERVAL '15 minutes', pickup_datetime) AS pickup_15mins_window,
        COUNT(*) as number_of_rides
    FROM taxi_clean
    GROUP BY pickup_15mins_window
    ORDER BY number_of_rides DESC;
""")

┌──────────────────────┬─────────────────┐
│ pickup_15mins_window │ number_of_rides │
│      timestamp       │      int64      │
├──────────────────────┼─────────────────┤
│ 2025-01-16 20:15:00  │            2409 │
│ 2025-01-16 18:00:00  │            2387 │
│ 2025-01-16 18:15:00  │            2380 │
│ 2025-01-16 17:45:00  │            2343 │
│ 2025-01-16 17:30:00  │            2323 │
│ 2025-01-16 18:30:00  │            2313 │
│ 2025-01-16 18:45:00  │            2222 │
│ 2025-01-16 20:30:00  │            2213 │
│ 2025-01-23 17:45:00  │            2180 │
│ 2025-01-08 18:00:00  │            2171 │
│          ·           │               · │
│          ·           │               · │
│          ·           │               · │
│ 2025-01-21 03:30:00  │              27 │
│ 2025-01-08 04:00:00  │              22 │
│ 2025-01-15 03:30:00  │              22 │
│ 2024-12-31 23:45:00  │               9 │
│ 2024-12-31 23:30:00  │               3 │
│ 2024-12-31 23:15:00  │               3 │
│ 2024-12-3

*Day-over-day revenue change*

In [152]:
con.sql("""
        WITH daily_revenue AS (
            SELECT 
                DATE(pickup_datetime) AS trip_date,
                SUM(fare_amount) AS revenue
            FROM taxi_clean
            GROUP BY trip_date
        ),
        revenue_with_lag AS (
            SELECT 
                trip_date,
                revenue,
                LAG(revenue) OVER (ORDER BY trip_date) as prev_revenue
            FROM daily_revenue
        )
        SELECT
            trip_date,
            ROUND(revenue, 2) AS revenue,
            ROUND(prev_revenue, 2) AS prev_revenue,
            ROUND(
                (revenue - prev_revenue) / prev_revenue * 100,
                2
            ) AS revenue_change_pct
        FROM revenue_with_lag
        ORDER BY trip_date ASC;
""")

┌────────────┬────────────┬──────────────┬────────────────────┐
│ trip_date  │  revenue   │ prev_revenue │ revenue_change_pct │
│    date    │   double   │    double    │       double       │
├────────────┼────────────┼──────────────┼────────────────────┤
│ 2024-12-31 │      401.6 │         NULL │               NULL │
│ 2025-01-01 │ 1461385.19 │        401.6 │          363790.73 │
│ 2025-01-02 │ 1583396.19 │   1461385.19 │               8.35 │
│ 2025-01-03 │ 1599759.38 │   1583396.19 │               1.03 │
│ 2025-01-04 │ 1655774.41 │   1599759.38 │                3.5 │
│ 2025-01-05 │ 1451625.47 │   1655774.41 │             -12.33 │
│ 2025-01-06 │ 1395774.39 │   1451625.47 │              -3.85 │
│ 2025-01-07 │ 1559595.17 │   1395774.39 │              11.74 │
│ 2025-01-08 │ 1639968.59 │   1559595.17 │               5.15 │
│ 2025-01-09 │ 1761703.63 │   1639968.59 │               7.42 │
│     ·      │      ·     │        ·     │                 ·  │
│     ·      │      ·     │        ·    